# Assignment 22: Embedding Models, Vector Stores & Similarity Search
**Student Name:** Parth Dadhaniya  
**Course:** GenAI Course Assignments  
**Role:** Generative AI Engineer  
**Scenario:** Personal Knowledge Search Engine (Core Retrieval Backbone for RAG)

---

### Overview
After loading and splitting documents, the core foundation of a RAG system is:
1. Converting text into embeddings (OpenAI, Hugging Face, Ollama).
2. Storing embeddings in vector databases (FAISS, ChromaDB).
3. Performing similarity search to retrieve relevant chunks for a question.


## Part 1: Embedding Models

### Setup: Load Sample Text Chunks
We load `notes.txt` from the `data/` folder and split it into chunks.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load and split sample text
docs = TextLoader("data/notes.txt", encoding="utf-8").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print("Loaded chunks:", len(chunks))


### Task 1: OpenAI Embedding Model
Using OpenAI `text-embedding-3-small` (1536 dimensions) to convert text into embeddings.


In [ ]:
# initialize openai embeddings
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-"):
    from langchain_openai import OpenAIEmbeddings
    model = OpenAIEmbeddings(model="text-embedding-3-small")
    sample_vec = model.embed_query(chunks[0].page_content)
else:
    # offline fallback matching 1536 dimensions of text-embedding-3-small
    import numpy as np
    np.random.seed(42)
    sample_vec = np.random.randn(1536).tolist()
    print("Note: OPENAI_API_KEY not found, using local 1536-dim vector for testing.")

print("Model: text-embedding-3-small")
print("Vector length (dimensions):", len(sample_vec))
print("Sample embedding values (first 5):")
print([round(x, 4) for x in sample_vec[:5]])


### Task 2: Hugging Face Embedding Model
Using Hugging Face `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions) for local, free embeddings.


In [ ]:
import time
from langchain_community.embeddings import HuggingFaceEmbeddings

# load local huggingface model
print("Loading Hugging Face model: all-MiniLM-L6-v2...")
t0 = time.time()
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print(f"Model loaded in {time.time() - t0:.2f} seconds.")

# embed first chunk
vec = model.embed_query(chunks[0].page_content)

print("
Model: sentence-transformers/all-MiniLM-L6-v2")
print("Vector length (dimensions):", len(vec))
print("Sample values (first 5):", [round(x, 4) for x in vec[:5]])

# test embedding multiple chunks
t1 = time.time()
batch_vecs = model.embed_documents([c.page_content for c in chunks[:3]])
print(f"
Embedded 3 chunks in {time.time() - t1:.3f} seconds.")
print("Each vector dimension:", len(batch_vecs[0]))


### Task 3: Comparison - OpenAI vs Hugging Face Embeddings

#### 1. When to prefer OpenAI embeddings?
- When building production cloud apps that need high accuracy across many languages.
- When you don't want to manage local GPUs or model weights.
- When 1536 or 3072 dimensions are needed for nuanced search.

#### 2. When to prefer Hugging Face embeddings?
- When data privacy is critical and documents cannot leave your local server.
- In completely offline or air-gapped systems.
- When indexing large volumes of documents where API costs would be too high.
- When you need domain-specific fine-tuned models (e.g. biobert, legal-bert).

#### 3. Cost vs Performance Trade-offs:
- **OpenAI:** Very fast setup with an API key, but ongoing per-token cost ($0.02 per 1M tokens).
- **Hugging Face:** Zero API fees, but requires local computer RAM and CPU/GPU power.


## Part 2: Document Similarity Search

### Task 4: Similarity Search App (Core Logic)
We load text and CSV data, embed them, and build a search function using NumPy cosine similarity.


In [ ]:
import numpy as np
from langchain_community.document_loaders import CSVLoader

# 1. Load documents
docs = TextLoader("data/notes.txt", encoding="utf-8").load()
docs.extend(CSVLoader("data/data.csv", encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print("Total chunks in index:", len(chunks))

# 2. Convert chunks to embeddings
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
chunk_texts = [c.page_content for c in chunks]
chunk_vectors = np.array(model.embed_documents(chunk_texts))

# normalize vectors for cosine similarity
chunk_vectors = chunk_vectors / np.linalg.norm(chunk_vectors, axis=1, keepdims=True)

# 3. Search function
def search(query, top_k=3):
    q_vec = np.array(model.embed_query(query))
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = np.dot(chunk_vectors, q_vec)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(chunks[idx], scores[idx]) for idx in top_indices]

# 4. Test queries
queries = [
    "What are the privacy and compliance rules?",
    "Who works on the backend engineer role?",
    "Why is text splitting needed for large documents?"
]

for q in queries:
    print(f"
Query: '{q}'")
    matches = search(q, top_k=2)
    for i, (chunk, score) in enumerate(matches, 1):
        src = chunk.metadata.get("source", "unknown")
        print(f"  Match {i} (score: {score:.3f}, source: {src}):")
        print("   ", chunk.page_content.strip()[:100].replace("
", " "), "...")


### Task 5: Similarity Search with LangChain Abstraction
Using LangChain's `vectorstore.similarity_search(query)` with ChromaDB.


In [ ]:
from langchain_community.vectorstores import Chroma

# 1. Create Chroma vectorstore
print("Creating in-memory Chroma vector store...")
vectorstore = Chroma.from_documents(chunks, model)

# 2. Search using vectorstore.similarity_search
query = "What are the rules for data privacy and security?"
print(f"
Query: '{query}'")

results = vectorstore.similarity_search(query, k=2)

print(f"
Retrieved {len(results)} documents:")
for i, doc in enumerate(results, 1):
    src = doc.metadata.get("source", "unknown")
    print(f"
Result {i} (source: {src}):")
    print("Content:", doc.page_content.strip()[:140].replace("
", " "), "...")


## Part 3: Ollama Embeddings (Local Setup)

### Task 6: Ollama Embedding Model
Ollama lets you run open-source models locally without any cloud connection.

#### How to run Ollama:
1. Install Ollama from [ollama.ai](https://ollama.ai)
2. Run in terminal: `ollama serve`
3. Pull model: `ollama pull nomic-embed-text`
4. Use endpoint: `http://localhost:11434/api/embeddings`


In [ ]:
import json
import urllib.request

text = "The Personal Knowledge Assistant helps team members query documents."

ollama_url = "http://localhost:11434/api/embeddings"
payload = json.dumps({"model": "nomic-embed-text", "prompt": text}).encode("utf-8")
req = urllib.request.Request(ollama_url, data=payload, headers={"Content-Type": "application/json"})

try:
    with urllib.request.urlopen(req, timeout=2) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        vector = data["embedding"]
        print("Connected to live Ollama server successfully!")
except Exception:
    np.random.seed(123)
    vector = np.random.randn(768).tolist()
    print("Ollama server not active on localhost:11434. Generated nomic-embed-text 768-dim format for testing.")

print("
Model: nomic-embed-text")
print("Vector dimensions:", len(vector))
print("Sample values (first 5):", [round(x, 4) for x in vector[:5]])


## Part 4: Vector Stores

### Task 7: FAISS Vector Store
Storing embeddings in FAISS, performing search, and saving the index to disk.


In [ ]:
from task7_faiss_vectorstore import faiss_cls

# 1. Store embeddings in FAISS
faiss_db = faiss_cls.from_documents(chunks, model)

# 2. Similarity search
query = "What is the policy on sensitive PII data?"
results = faiss_db.similarity_search(query, k=2)

print(f"Query: '{query}'")
for i, doc in enumerate(results, 1):
    print(f"Match {i}:", doc.page_content.strip()[:100].replace("
", " "), "...")

# 3. Save index to disk
faiss_db.save_local("faiss_index")


### Task 8: ChromaDB Vector Store
Creating a persistent ChromaDB database on disk in `chroma_db/`.


In [ ]:
import shutil

# clean old directory to test fresh
if os.path.exists("chroma_db"):
    try:
        shutil.rmtree("chroma_db")
    except Exception:
        pass

# Create and persist ChromaDB
print("Creating persistent ChromaDB collection in 'chroma_db'...")
db = Chroma.from_documents(
    documents=chunks,
    embedding=model,
    persist_directory="chroma_db"
)
print("Saved document embeddings to ChromaDB.")

# Similarity search
query = "Who is the DevOps Cloud Architect?"
results = db.similarity_search(query, k=2)

print(f"
Query: '{query}'")
for i, doc in enumerate(results, 1):
    src = doc.metadata.get("source", "unknown")
    print(f"Result {i} (source: {src}):", doc.page_content.strip()[:100].replace("
", " "), "...")


### Task 9: FAISS vs ChromaDB Comparison

#### 1. In-Memory vs Persistent Storage:
- **FAISS:** Loads vectors in RAM for fast search, but does not save automatically to disk unless you call `save_local()`.
- **ChromaDB:** Automatically persists embeddings, text chunks, and metadata to disk using SQLite/DuckDB.

#### 2. When to Use FAISS?
- For very large vector datasets (millions or billions of items) needing GPU acceleration.
- For quick in-memory lookups during a temporary script or session.
- When you only need raw distance search without database features.

#### 3. When to Use ChromaDB?
- For complete RAG applications needing persistent local storage.
- When you need to filter results by metadata (like department, date, or author).
- When you want built-in collection management (add, update, delete documents easily).


## Part 5: Mini Project Integration

### Task 10: End-to-End Similarity Search Pipeline
Pipeline: `Documents -> Splitter -> Embeddings -> Vector Store -> Similarity Search` supporting switchable backends.


In [ ]:
from task10_similarity_pipeline import run_similarity_pipeline

# Test 1: Hugging Face + ChromaDB
run_similarity_pipeline("Who is the NLP specialist?", "huggingface", "chroma")

# Test 2: OpenAI style + FAISS
run_similarity_pipeline("What are the compliance rules?", "openai", "faiss")


### Task 11: Observations & Insights

#### 1. Importance of Embeddings in GenAI:
- Converts words and sentences into numbers (vectors) that computers can compare mathematically.
- Captures semantic meaning so that synonyms and related concepts match even if different words are used.
- Enables calculating cosine similarity between a user question and document chunks.

#### 2. Why Vector Databases are Required:
- Regular SQL databases are made for exact matching (LIKE, =), which cannot find similar meanings.
- Vector databases use Approximate Nearest Neighbor (ANN) search to search through thousands or millions of vectors in just a few milliseconds.
- They allow combining vector similarity with metadata filters (like filtering by date or department).

#### 3. How This Pipeline Enables RAG Systems:
- **Ingestion:** Load company documents and split them into chunks.
- **Indexing:** Convert chunks to embeddings and store in a vector database.
- **Retrieval:** When a user asks a question, find the most relevant chunks using similarity search.
- **Generation:** Pass those relevant chunks to the LLM as context so it gives accurate, factual answers.
